In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG   = "clutchlytics"
SCHEMA    = "bronze"
VOLUME    = "nhl_game_summaries"
TABLE     = f"{CATALOG}.{SCHEMA}.raw_nhl_game_summaries"
 
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
 
print(f"Source : {VOLUME_PATH}")
print(f"Target : {TABLE}")

In [0]:
# ── DISCOVER FILES ────────────────────────────────────────────────────────────
 
all_files    = dbutils.fs.ls(VOLUME_PATH)
summary_files = [f for f in all_files if f.name.endswith(".json")]
 
print(f"Summary files found: {len(summary_files)}")
for f in sorted(summary_files, key=lambda x: x.name):
    print(f"  {f.name:<40} {f.size:>10,} bytes")

In [0]:
# ── READ + PARSE ALL FILES ────────────────────────────────────────────────────
 
import json
from datetime import datetime, timezone
 
ingested_at = datetime.now(timezone.utc).isoformat()
rows        = []
skipped     = []
 
for f in summary_files:
    filename = f.name  # e.g. 401869715_summary.json
 
    # ── Parse event_id from filename ──
    try:
        event_id = filename.replace("_summary.json", "")
        if not event_id.isdigit():
            raise ValueError(f"event_id not numeric: {event_id}")
    except Exception as e:
        skipped.append((filename, str(e)))
        continue
 
    # ── Read and parse JSON ──
    try:
        raw_text = spark.read.text(f.path)
        json_str = "\n".join([row.value for row in raw_text.collect()])
        payload  = json.loads(json_str)
    except Exception as e:
        skipped.append((filename, f"parse error: {e}"))
        continue
 
    # ── Extract meta envelope ──
    meta        = payload.get("meta", {})
    pulled_at   = meta.get("pulled_at")
    home_team   = meta.get("home_team")
    away_team   = meta.get("away_team")
    home_score  = meta.get("home_score")
    away_score  = meta.get("away_score")
    game_date   = meta.get("game_date")
    season      = meta.get("season")
    season_type = meta.get("season_type")
    round_num   = meta.get("round")
    source_url  = meta.get("source_url")
 
    # ── Extract data block ──
    data = payload.get("data", {})
 
    # ── gameInfo — attendance + officials ──
    game_info  = data.get("gameInfo", {})
    attendance = game_info.get("attendance")
    officials  = json.dumps(game_info.get("officials", []))
 
    # ── boxscore — store full block as JSON string ──
    # Silver extracts: teams stats, player additive stats, scratched flags
    boxscore_json = json.dumps(data.get("boxscore", {}))
 
    # ── odds — store full block as JSON string ──
    # May be empty list for some games — Silver handles nulls
    odds_json = json.dumps(data.get("odds", []))
 
    # ── Quick count of teams and players in boxscore for validation ──
    boxscore     = data.get("boxscore", {})
    team_count   = len(boxscore.get("teams", []))
    player_count = sum(
        len(stat_group.get("athletes", []))
        for team_players in boxscore.get("players", [])
        for stat_group in team_players.get("statistics", [])
    )
 
    rows.append({
        # ── Game identity (from meta) ──
        "event_id":       event_id,
        "home_team":      home_team,
        "away_team":      away_team,
        "home_score":     home_score,
        "away_score":     away_score,
        "game_date":      game_date,
        "season":         season,
        "season_type":    season_type,
        "round":          round_num,
 
        # ── gameInfo ──
        "attendance":     attendance,
        "officials_json": officials,
 
        # ── Raw JSON blocks (parsed in Silver) ──
        "boxscore_json":  boxscore_json,
        "odds_json":      odds_json,
 
        # ── Validation counts ──
        "boxscore_team_count":   team_count,
        "boxscore_player_count": player_count,
 
        # ── Ingestion metadata ──
        "source_file":    filename,
        "source_url":     source_url,
        "pulled_at":      pulled_at,
        "ingested_at":    ingested_at,
    })
 
    print(f"  {filename:<40} → teams: {team_count}  players: {player_count}  "
          f"attendance: {attendance}")
 
print(f"\nRows built  : {len(rows)}")
print(f"Skipped     : {len(skipped)}")
 
if skipped:
    print("\nSkipped files:")
    for fname, reason in skipped:
        print(f"  {fname}: {reason}")

In [0]:
# ── SPOT CHECK — verify boxscore and odds blocks on one game ──────────────────
 
sample = rows[0] if rows else None
 
if sample:
    print(f"Spot check — {sample['away_team']} @ {sample['home_team']}")
    print(f"  event_id   : {sample['event_id']}")
    print(f"  game_date  : {sample['game_date']}")
    print(f"  score      : {sample['away_score']} - {sample['home_score']}")
    print(f"  attendance : {sample['attendance']}")
    print(f"  team_count : {sample['boxscore_team_count']}")
    print(f"  player_count: {sample['boxscore_player_count']}")
 
    # Peek at team stats from boxscore
    boxscore = json.loads(sample["boxscore_json"])
    teams    = boxscore.get("teams", [])
    if teams:
        print(f"\n  Team stats sample ({teams[0].get('team', {}).get('abbreviation', '')}):")
        for stat in teams[0].get("statistics", []):
            print(f"    {stat.get('name'):<30} = {stat.get('displayValue')}")
 
    # Peek at odds
    odds = json.loads(sample["odds_json"])
    print(f"\n  Odds block type : {type(odds).__name__}")
    print(f"  Odds length     : {len(odds) if isinstance(odds, list) else 'n/a'}")
    if isinstance(odds, list) and odds:
        print(f"  Odds[0] keys    : {list(odds[0].keys())}")

In [0]:
# ── WRITE TO DELTA ────────────────────────────────────────────────────────────
# MERGE on event_id — re-upload of same file overwrites row with latest data.
 
if not rows:
    raise ValueError("No rows built — check file parsing above before writing.")
 
summaries_df = spark.createDataFrame(rows)
 
table_exists = spark.catalog.tableExists(TABLE)
 
if not table_exists:
    (
        summaries_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TABLE)
    )
    print(f"Table created: {TABLE}")
    print(f"Rows written : {len(rows)}")
 
else:
    summaries_df.createOrReplaceTempView("new_summaries")
 
    spark.sql(f"""
        MERGE INTO {TABLE} AS target
        USING new_summaries AS source
        ON target.event_id = source.event_id
        WHEN MATCHED THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """)
    print(f"Merged into existing table: {TABLE}")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
result = spark.sql(f"""
    SELECT
        event_id,
        away_team,
        home_team,
        away_score,
        home_score,
        game_date,
        round,
        attendance,
        boxscore_team_count,
        boxscore_player_count,
        pulled_at
    FROM {TABLE}
    ORDER BY game_date
""")
 
total = result.count()
print(f"Total rows in {TABLE}: {total}")
result.show(50, truncate=False)

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                                    AS total_games,
        COUNT(DISTINCT round)                                       AS rounds,
        COUNT(CASE WHEN event_id IS NULL    THEN 1 END)            AS null_event_ids,
        COUNT(CASE WHEN home_team IS NULL   THEN 1 END)            AS null_home_teams,
        COUNT(CASE WHEN away_team IS NULL   THEN 1 END)            AS null_away_teams,
        COUNT(CASE WHEN attendance IS NULL  THEN 1 END)            AS null_attendance,
        COUNT(CASE WHEN boxscore_team_count != 2 THEN 1 END)       AS bad_team_count,
        COUNT(CASE WHEN boxscore_player_count = 0 THEN 1 END)      AS empty_player_blocks,
        COUNT(CASE WHEN odds_json = '[]'    THEN 1 END)            AS games_no_odds,
        AVG(boxscore_player_count)                                  AS avg_players_per_game,
        AVG(attendance)                                             AS avg_attendance,
        MIN(game_date)                                              AS earliest_game,
        MAX(game_date)                                              AS latest_game
    FROM {TABLE}
""")
 
print("Sanity checks:")
checks.show(truncate=False)